# 03. 로그 전이와 메트릭 점수 결합

목표: raw log를 간단한 template로 정규화하고, 정상 sequence에서 드문 전이에 높은 surprise score를 부여한 뒤 metric anomaly score와 결합합니다.

여기서 사용하는 정규식 normalizer는 학습용이며 Drain3 구현이 아닙니다. production에서는 Drain3의 masking, persistence와 cluster 제한을 사용하세요.

In [ ]:
import math
import random
import re
from collections import Counter, defaultdict

random.seed(19)

MASKS = [
    (re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"), "<IP>"),
    (re.compile(r"\b[0-9a-f]{8}-[0-9a-f-]{27,}\b", re.I), "<UUID>"),
    (re.compile(r"\b\d+\b"), "<NUM>"),
]

def normalize_log(message):
    template = message
    for pattern, replacement in MASKS:
        template = pattern.sub(replacement, template)
    return template

examples = [
"INFO response sent status=200 request=1250",
"INFO response sent status=200 request=9981",
"ERROR database timeout host=10.2.4.8 retry=3",
]
for message in examples:
    print(message, "→", normalize_log(message))

## 합성 로그와 메트릭 score

정상은 request → DB query → response 순서를 반복합니다. 장애 구간에는 database timeout과 retry가 나타납니다. 점검 로그는 희귀하지만 metric은 정상이므로 paging하지 않는 것이 목표입니다.

In [ ]:
def build_timeline(size=260):
    normal_messages = (
        "INFO request accepted id={n}",
        "INFO database query completed rows={n}",
        "INFO response sent status=200 request={n}",
    )
    rows = []
    for t in range(size):
        message = normal_messages[t % len(normal_messages)].format(n=1000 + t)
        metric_score = 0.08 + random.random() * 0.10
        incident = False

        if 160 <= t < 167:
            incident_messages = (
                "ERROR database timeout host=10.2.4.8 retry=1",
                "WARN retry scheduled attempt=2",
                "ERROR request failed status=503 id=7788",
            )
            message = incident_messages[(t - 160) % len(incident_messages)]
            metric_score = 0.88 + random.random() * 0.10
            incident = True

        # 단발성 metric spike는 persistence로 억제합니다.
        if t == 200:
            metric_score = 0.91

        # 희귀하지만 정상인 maintenance event입니다.
        if 220 <= t < 223:
            message = f"INFO maintenance task started ticket={9000 + t}"
            metric_score = 0.10

        rows.append({
            "t": t,
            "message": message,
            "template": normalize_log(message),
            "metric_score": metric_score,
            "is_incident": incident,
        })
    return rows

timeline = build_timeline()
print("rows:", len(timeline), "incident points:", sum(row["is_incident"] for row in timeline))

## 정상 전이 모델

첫 120개 관측을 정상 baseline으로 사용합니다. Laplace smoothing으로 처음 보는 전이도 0 probability가 되지 않게 합니다.

In [ ]:
training_end = 120
transition_counts = defaultdict(Counter)
templates = {row["template"] for row in timeline[:training_end]}

for previous, current in zip(timeline[:training_end - 1], timeline[1:training_end]):
    transition_counts[previous["template"]][current["template"]] += 1

def transition_surprise(previous_template, current_template, alpha=1.0):
    counts = transition_counts[previous_template]
    vocabulary_size = len(templates) + 1  # unseen template bucket
    probability = (counts[current_template] + alpha) / (sum(counts.values()) + alpha * vocabulary_size)
    return -math.log(probability)

known = timeline[0]["template"], timeline[1]["template"]
print("known transition surprise:", round(transition_surprise(*known), 3))
print("unseen transition surprise:", round(transition_surprise(known[0], "ERROR unseen failure"), 3))

## Score fusion과 persistence

희귀 로그 하나만으로 paging하지 않습니다. metric과 log score가 함께 높거나 결합 점수가 높고, 최근 3회 중 2회 이상 지속될 때 incident alert를 확정합니다.

In [ ]:
raw_candidates = [False] * training_end
explanations = [None] * training_end

for index in range(training_end, len(timeline)):
    previous = timeline[index - 1]
    current = timeline[index]
    raw_surprise = transition_surprise(previous["template"], current["template"])
    log_score = min(raw_surprise / 6.0, 1.0)
    metric_score = current["metric_score"]
    fused_score = 0.65 * metric_score + 0.35 * log_score
    candidate = fused_score >= 0.65 or (metric_score >= 0.75 and log_score >= 0.40)
    raw_candidates.append(candidate)
    explanations.append({
        "metric": metric_score,
        "log": log_score,
        "fused": fused_score,
        "template": current["template"],
    })

def persistence_gate(flags, window=3, required=2):
    result = []
    for index in range(len(flags)):
        recent = flags[max(0, index - window + 1):index + 1]
        result.append(sum(recent) >= required)
    return result

alerts = persistence_gate(raw_candidates)
for row, alert, explanation in zip(timeline, alerts, explanations):
    if alert:
        print(
            f"t={row['t']} metric={explanation['metric']:.2f} "
            f"log={explanation['log']:.2f} fused={explanation['fused']:.2f} "
            f"template={explanation['template']}"
        )

In [ ]:
labels = [row["is_incident"] for row in timeline]
tp = sum(label and alert for label, alert in zip(labels, alerts))
fp = sum((not label) and alert for label, alert in zip(labels, alerts))
fn = sum(label and (not alert) for label, alert in zip(labels, alerts))
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0

maintenance_alerts = [row["t"] for row, alert in zip(timeline, alerts) if 220 <= row["t"] < 223 and alert]
print({"tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 3), "recall": round(recall, 3)})
print("maintenance alerts:", maintenance_alerts)
print("single metric spike alerted:", alerts[200])

실제 운영에서는 전이 모델도 무제한 online 학습시키지 마세요. 새 배포나 공격 sequence가 정상으로 흡수되지 않도록 지연 학습, 승인된 retraining, `max_clusters`, drift 감시와 model versioning이 필요합니다.